# CT Liver Tumour Pipeline

**Before running:**
1. Attach your two Kaggle Datasets in the right-hand panel under **Add data**:
   - `liver-ct-code`  — all your `.py` source files
   - `liver-ct-batch1` — your `Training_Batch_1/` CT data
2. Enable GPU: right panel → **Session options → Accelerator → GPU T4**
3. Update `DATA_DATASET_SLUG` and `CODE_DATASET_SLUG` in Cell 2 to match your exact dataset names.

**Stage gating** is controlled in Cell 7. Set each stage to `True` or `False`.

## Cell 1 — Install dependencies

In [ ]:
# einops is the only non-standard dependency (used in models.py AttentionBlock)
# All other packages (torch, numpy, cv2, matplotlib, tqdm) are pre-installed on Kaggle
!pip install einops -q
print('Dependencies ready.')

## Cell 2 — Configure dataset slugs

Set these to match the names of your Kaggle Datasets exactly.
The slug is the last part of the dataset URL:
`kaggle.com/datasets/yourname/liver-ct-batch1` → slug is `liver-ct-batch1`

In [ ]:
# ── CHANGE THESE to match your actual Kaggle Dataset names ────────────────
DATA_DATASET_SLUG = 'Diffusion_training_batch1'   # dataset containing Training_Batch_1/
CODE_DATASET_SLUG = 'Diff4MMLits'     # dataset containing all your .py files
# ──────────────────────────────────────────────────────────────────────────

from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')
WORKING      = Path('/kaggle/working')

DATA_INPUT = KAGGLE_INPUT / DATA_DATASET_SLUG
CODE_INPUT = KAGGLE_INPUT / CODE_DATASET_SLUG

print(f'Data input path : {DATA_INPUT}')
print(f'Code input path : {CODE_INPUT}')
print(f'Working dir     : {WORKING}')

## Cell 3 — Verify data and code datasets are accessible

In [ ]:
import os

# ── Verify code dataset ───────────────────────────────────────────────────
print('=== Code dataset ===')
if not CODE_INPUT.exists():
    raise FileNotFoundError(
        f"Code dataset not found at {CODE_INPUT}.\n"
        f"Make sure you attached '{CODE_DATASET_SLUG}' in the right-hand Add data panel."
    )

py_files = list(CODE_INPUT.glob('*.py'))
if not py_files:
    raise FileNotFoundError(
        f"No .py files found in {CODE_INPUT}.\n"
        f"Check that your code dataset was uploaded correctly."
    )

required_modules = [
    'main_kaggle.py', 'models.py', 'train.py', 'segmentation.py',
    'preprocess.py',  'inpainting.py', 'dataset.py', 'utils.py', 'visualise.py'
]
found_names = {f.name for f in py_files}
missing     = [m for m in required_modules if m not in found_names]

if missing:
    print(f'WARNING — these expected files are missing from the code dataset:')
    for m in missing:
        print(f'  missing: {m}')
else:
    print('All required .py files found.')

print(f'Files in code dataset ({len(py_files)} .py files):')
for f in sorted(py_files):
    print(f'  {f.name}')

# ── Verify data dataset ───────────────────────────────────────────────────
print()
print('=== Data dataset ===')
if not DATA_INPUT.exists():
    raise FileNotFoundError(
        f"Data dataset not found at {DATA_INPUT}.\n"
        f"Make sure you attached '{DATA_DATASET_SLUG}' in the right-hand Add data panel."
    )

# Show top-level contents of the data dataset
top_level = sorted(DATA_INPUT.iterdir())
print(f'Contents of {DATA_INPUT}:')
for item in top_level:
    if item.is_dir():
        n_files = len(list(item.rglob('*')))
        print(f'  {item.name}/  ({n_files} files inside)')
    else:
        print(f'  {item.name}')

# Check Training_Batch_1 specifically
training_dir = DATA_INPUT / 'Training_Batch_1'
if not training_dir.exists():
    # Maybe the folder structure is flat — show what's there to help diagnose
    print(f'WARNING — Training_Batch_1/ not found inside {DATA_INPUT}.')
    print('Update DATA_DATASET_SLUG or the DATASET_IN path in main_kaggle.py.')
else:
    volumes = list(training_dir.iterdir())
    print(f'Training_Batch_1/ found — {len(volumes)} items inside.')

## Cell 4 — Copy source files to /kaggle/working/ and add to Python path

Kaggle's `/kaggle/input/` is read-only. We copy all `.py` files to `/kaggle/working/`
so they can import each other (relative imports require files to be in the same directory).

In [ ]:
import shutil
import sys

# Copy all .py files from the code dataset to /kaggle/working/
print('Copying source files to /kaggle/working/ ...')
for py_file in sorted(CODE_INPUT.glob('*.py')):
    dest = WORKING / py_file.name
    shutil.copy(py_file, dest)
    print(f'  {py_file.name} → {dest}')

# Add /kaggle/working/ to the front of sys.path
# This must come BEFORE any imports from your modules
working_str = str(WORKING)
if working_str not in sys.path:
    sys.path.insert(0, working_str)
    print(f'\nAdded to sys.path: {working_str}')
else:
    print(f'\nAlready in sys.path: {working_str}')

print('\nsys.path (first 5 entries):')
for p in sys.path[:5]:
    print(f'  {p}')

## Cell 5 — Verify GPU

In [ ]:
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {props.total_memory / 1e9:.1f} GB')
    print(f'CUDA version    : {torch.version.cuda}')
    # Quick smoke-test: allocate a small tensor on GPU
    x = torch.ones(3, 3, device='cuda')
    print(f'Smoke test      : tensor on {x.device} — OK')
    del x
    torch.cuda.empty_cache()
else:
    print('WARNING — No GPU detected. Enable GPU in Session options → Accelerator.')

## Cell 6 — Patch main_kaggle.py with correct dataset slugs

This updates the `KAGGLE_DATASET_SLUG` constant in the copied `main_kaggle.py`
to match what you set in Cell 2, so you only need to change it in one place.

In [ ]:
main_path = WORKING / 'main_kaggle.py'
content   = main_path.read_text()

# Replace whatever slug is in the file with the one set in Cell 2
import re
content = re.sub(
    r'KAGGLE_DATASET_SLUG\s*=\s*["\'][^"\']*["\']',
    f'KAGGLE_DATASET_SLUG = "{DATA_DATASET_SLUG}"',
    content
)
main_path.write_text(content)

# Confirm the patch
for line in content.splitlines():
    if 'KAGGLE_DATASET_SLUG' in line and '=' in line and '#' not in line.split('=')[0]:
        print(f'Patched: {line.strip()}')
        break

print('main_kaggle.py patched successfully.')

## Cell 7 — Configure which stages to run

Set each stage to `True` or `False` before running Cell 8.

| Stage | What it does | Run first time? | Run after? |
|-------|-------------|-----------------|------------|
| 1 — Preprocessing | Raw NIfTI → processed_volumes/ | ✅ Yes | ❌ No — outputs persist |
| 2 — Inpainting | CT + mask → tumour-free CT | ✅ Yes | ❌ No — outputs persist |
| 3 — Segmentation | Train liver/tumour segmenter | ✅ Yes | ✅ To resume training |
| 4 — Diffusion | Train diffusion model | ✅ Yes | ✅ To resume training |

In [ ]:
# ── Set stages here ───────────────────────────────────────────────────────
RUN_STAGE_1_PREPROCESS  = False   # set True on very first run only
RUN_STAGE_2_INPAINTING  = False   # set True after Stage 1 completes
RUN_STAGE_3_SEGMENTATION = True   # active
RUN_STAGE_4_DIFFUSION   = False   # set True after Stage 3 completes
# ──────────────────────────────────────────────────────────────────────────

# Patch the stage gates in the copied main_kaggle.py
content = main_path.read_text()

def _patch_stage(text, fn_name, enabled):
    """Replace  if True/False:\n        fn_name()  in the main() block."""
    val = 'True' if enabled else 'False'
    # Match the if True/False line immediately before the function call
    pattern = rf'(if )(?:True|False)(:\s*\n\s+{re.escape(fn_name)}\(\))'
    return re.sub(pattern, rf'\g<1>{val}\2', text)

content = _patch_stage(content, 'run_preprocessing',        RUN_STAGE_1_PREPROCESS)
content = _patch_stage(content, 'run_inpainting',           RUN_STAGE_2_INPAINTING)
content = _patch_stage(content, 'run_segmentation_training',RUN_STAGE_3_SEGMENTATION)
content = _patch_stage(content, 'run_diffusion_training',   RUN_STAGE_4_DIFFUSION)
main_path.write_text(content)

print('Stage configuration:')
print(f'  Stage 1 — Preprocessing  : {"ON" if RUN_STAGE_1_PREPROCESS   else "off"}')
print(f'  Stage 2 — Inpainting     : {"ON" if RUN_STAGE_2_INPAINTING    else "off"}')
print(f'  Stage 3 — Segmentation   : {"ON" if RUN_STAGE_3_SEGMENTATION  else "off"}')
print(f'  Stage 4 — Diffusion      : {"ON" if RUN_STAGE_4_DIFFUSION      else "off"}')

## Cell 8 — Restore checkpoints from a previous session (optional)

If you have checkpoints from a previous session saved as a Kaggle Dataset,
attach that dataset and set the slug below. Skip this cell if starting fresh.

In [ ]:
# Set to None to skip, or set to your checkpoint dataset slug to restore
CHECKPOINT_DATASET_SLUG = None    # e.g. 'liver-ct-checkpoints'

if CHECKPOINT_DATASET_SLUG is not None:
    ckpt_input = KAGGLE_INPUT / CHECKPOINT_DATASET_SLUG
    if not ckpt_input.exists():
        print(f'WARNING — Checkpoint dataset not found at {ckpt_input}. Skipping restore.')
    else:
        restored = 0
        for pt_file in ckpt_input.rglob('*.pt'):
            # Preserve subfolder structure (seg_checkpoints/ vs checkpoints/)
            rel      = pt_file.relative_to(ckpt_input)
            dest     = WORKING / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(pt_file, dest)
            print(f'  Restored: {rel}')
            restored += 1
        print(f'\nRestored {restored} checkpoint file(s) to /kaggle/working/')
else:
    print('No checkpoint dataset specified — starting fresh.')
    print('(Set CHECKPOINT_DATASET_SLUG above to resume a previous session.)')

## Cell 9 — Run the pipeline

In [ ]:
import importlib

# Reload in case Cell 7 patched the file after a previous import
if 'main_kaggle' in sys.modules:
    importlib.reload(sys.modules['main_kaggle'])

import main_kaggle
main_kaggle.main()

## Cell 10 — Save checkpoints to output (run at end of session)

Kaggle exposes `/kaggle/working/` as the notebook **Output** after a session ends.
This cell lists everything that will be saved so you can verify before the session
times out. Download the `.pt` files or save them to a new Kaggle Dataset to
resume in a future session.

In [ ]:
import os

print('=== Checkpoint files that will be saved to Output ===')
total_bytes = 0
for pt in sorted(WORKING.rglob('*.pt')):
    rel   = pt.relative_to(WORKING)
    size  = pt.stat().st_size
    total_bytes += size
    print(f'  {rel}  ({size / 1e6:.1f} MB)')

print(f'\nTotal checkpoint size: {total_bytes / 1e6:.1f} MB')
print()
print('=== Visualisation files ===')
for png in sorted(WORKING.rglob('*.png')):
    rel = png.relative_to(WORKING)
    print(f'  {rel}')

print()
print('These files will appear in the Output tab when this notebook is saved.')
print('Download them or attach them as a new Kaggle Dataset to resume next session.')